In [ ]:
%pip install plyer

In [1]:
import os
import smtplib
import pandas as pd

from pathlib import Path
from datetime import datetime
from email.message import EmailMessage

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
TRANSACTION_FILE = Path(
    "kafka_transaction_results.csv"
)

ALERT_HISTORY_FILE = Path(
    "fraud_alert_history.csv"
)

SEND_DESKTOP_NOTIFICATION = True
SEND_EMAIL_NOTIFICATION = False

print("Notification settings configured!")

Notification settings configured!


In [3]:
SEND_EMAIL_NOTIFICATION = False

In [4]:
if not TRANSACTION_FILE.exists():
    raise FileNotFoundError(
        f"File not found: {TRANSACTION_FILE}"
    )

transaction_data = pd.read_csv(
    TRANSACTION_FILE
)

required_columns = [
    "stream_id",
    "fraud_probability",
    "risk_score",
    "risk_level",
    "predicted_fraud",
    "recommended_action"
]

missing_columns = [
    column
    for column in required_columns
    if column not in transaction_data.columns
]

if missing_columns:
    raise ValueError(
        "Missing columns: "
        + str(missing_columns)
    )

print(
    "Transactions loaded:",
    len(transaction_data)
)

transaction_data.head()

Transactions loaded: 100


,stream_id,produced_at,processed_at,fraud_probability,risk_score,risk_level,predicted_fraud,recommended_action
0,1,2026-08-15T00:37:24.527777,2026-08-15T01:05:09.028245,0.369077,36.91,Medium,0,Monitor transaction
1,2,2026-08-15T00:37:24.732644,2026-08-15T01:05:09.130719,0.197998,19.80,Low,0,Approve transaction
2,3,2026-08-15T00:37:24.934066,2026-08-15T01:05:09.217359,0.336204,33.62,Medium,0,Monitor transaction
3,4,2026-08-15T00:37:25.137062,2026-08-15T01:05:09.307878,0.250450,25.05,Low,0,Approve transaction
4,5,2026-08-15T00:37:25.339670,2026-08-15T01:05:09.381159,0.234259,23.43,Low,0,Approve transaction


In [5]:
def create_alert_type(row):

    if row["risk_level"] == "Critical":
        return "Critical Fraud Alert"

    if row["risk_level"] == "High":
        return "High-Risk Fraud Alert"

    if row["risk_level"] == "Medium":
        return "Monitoring Alert"

    return "No Alert"


def create_alert_message(row):

    return (
        f"Transaction ID: {row['stream_id']}\n"
        f"Risk score: {row['risk_score']:.2f}\n"
        f"Fraud probability: "
        f"{row['fraud_probability']:.4f}\n"
        f"Risk level: {row['risk_level']}\n"
        f"Recommended action: "
        f"{row['recommended_action']}"
    )


transaction_data["alert_type"] = (
    transaction_data.apply(
        create_alert_type,
        axis=1
    )
)

transaction_data["alert_message"] = (
    transaction_data.apply(
        create_alert_message,
        axis=1
    )
)

transaction_data[
    [
        "stream_id",
        "risk_level",
        "alert_type"
    ]
].head()

,stream_id,risk_level,alert_type
0,1,Medium,Monitoring Alert
1,2,Low,No Alert
2,3,Medium,Monitoring Alert
3,4,Low,No Alert
4,5,Low,No Alert


In [6]:
alert_data = transaction_data[
    transaction_data["risk_level"].isin(
        [
            "Critical",
            "High",
            "Medium"
        ]
    )
].copy()

alert_data = alert_data.sort_values(
    by="risk_score",
    ascending=False
).reset_index(drop=True)

print(
    "Transactions requiring notification:",
    len(alert_data)
)

alert_data[
    [
        "stream_id",
        "risk_score",
        "risk_level",
        "alert_type"
    ]
].head(10)

Transactions requiring notification: 29


,stream_id,risk_score,risk_level,alert_type
0,43,40.08,Medium,Monitoring Alert
1,89,39.53,Medium,Monitoring Alert
2,79,37.56,Medium,Monitoring Alert
3,1,36.91,Medium,Monitoring Alert
4,62,36.90,Medium,Monitoring Alert
5,60,36.34,Medium,Monitoring Alert
6,33,36.15,Medium,Monitoring Alert
7,93,35.30,Medium,Monitoring Alert
8,56,34.69,Medium,Monitoring Alert
9,12,34.58,Medium,Monitoring Alert


In [7]:
if ALERT_HISTORY_FILE.exists():

    alert_history = pd.read_csv(
        ALERT_HISTORY_FILE
    )

else:

    alert_history = pd.DataFrame(
        columns=[
            "alert_id",
            "stream_id",
            "alert_time",
            "alert_type",
            "risk_score",
            "risk_level",
            "notification_status"
        ]
    )

print(
    "Previous alerts:",
    len(alert_history)
)

Previous alerts: 29


In [8]:
def send_desktop_alert(
    title,
    message
):
    try:

        from plyer import notification

        notification.notify(
            title=title,
            message=message,
            app_name="Fraud Detection System",
            timeout=8
        )

        return "Desktop notification sent"

    except Exception as error:

        print(
            "Desktop notification unavailable:",
            error
        )

        return "Desktop notification failed"

In [9]:
EMAIL_SENDER = os.getenv(
    "FRAUD_EMAIL_SENDER"
)

EMAIL_PASSWORD = os.getenv(
    "FRAUD_EMAIL_PASSWORD"
)

EMAIL_RECEIVER = os.getenv(
    "FRAUD_EMAIL_RECEIVER"
)

print(
    "Email configured:",
    all(
        [
            EMAIL_SENDER,
            EMAIL_PASSWORD,
            EMAIL_RECEIVER
        ]
    )
)

Email configured: False


In [10]:
def send_email_alert(
    subject,
    message
):
    if not all(
        [
            EMAIL_SENDER,
            EMAIL_PASSWORD,
            EMAIL_RECEIVER
        ]
    ):
        return "Email settings unavailable"

    try:

        email_message = EmailMessage()

        email_message["Subject"] = subject
        email_message["From"] = EMAIL_SENDER
        email_message["To"] = EMAIL_RECEIVER

        email_message.set_content(message)

        with smtplib.SMTP_SSL(
            "smtp.gmail.com",
            465
        ) as smtp_server:

            smtp_server.login(
                EMAIL_SENDER,
                EMAIL_PASSWORD
            )

            smtp_server.send_message(
                email_message
            )

        return "Email notification sent"

    except Exception as error:

        print("Email notification failed:", error)

        return "Email notification failed"

In [11]:
new_alert_records = []

if "stream_id" in alert_history.columns:

    previously_alerted_ids = set(
        alert_history["stream_id"]
        .astype(str)
        .tolist()
    )

else:

    previously_alerted_ids = set()


for _, row in alert_data.iterrows():

    stream_id = str(row["stream_id"])

    if stream_id in previously_alerted_ids:

        print(
            f"Transaction {stream_id} "
            "was already notified."
        )

        continue

    title = row["alert_type"]
    message = row["alert_message"]

    notification_results = []

    print("\n" + "=" * 50)
    print(title)
    print(message)

    if SEND_DESKTOP_NOTIFICATION:

        desktop_status = send_desktop_alert(
            title,
            message
        )

        notification_results.append(
            desktop_status
        )

    if (
        SEND_EMAIL_NOTIFICATION
        and row["risk_level"]
        in ["High", "Critical"]
    ):

        email_status = send_email_alert(
            title,
            message
        )

        notification_results.append(
            email_status
        )

    if not notification_results:
        notification_results.append(
            "Console notification created"
        )

    alert_record = {
        "alert_id": (
            "ALERT-"
            + datetime.now().strftime(
                "%Y%m%d%H%M%S%f"
            )
        ),
        "stream_id": row["stream_id"],
        "alert_time": datetime.now().isoformat(),
        "alert_type": title,
        "risk_score": row["risk_score"],
        "risk_level": row["risk_level"],
        "notification_status": "; ".join(
            notification_results
        )
    }

    new_alert_records.append(
        alert_record
    )

Transaction 43 was already notified.
Transaction 89 was already notified.
Transaction 79 was already notified.
Transaction 1 was already notified.
Transaction 62 was already notified.
Transaction 60 was already notified.
Transaction 33 was already notified.
Transaction 93 was already notified.
Transaction 56 was already notified.
Transaction 12 was already notified.
Transaction 70 was already notified.
Transaction 3 was already notified.
Transaction 49 was already notified.
Transaction 22 was already notified.
Transaction 45 was already notified.
Transaction 67 was already notified.
Transaction 87 was already notified.
Transaction 31 was already notified.
Transaction 29 was already notified.
Transaction 96 was already notified.
Transaction 8 was already notified.
Transaction 59 was already notified.
Transaction 9 was already notified.
Transaction 98 was already notified.
Transaction 99 was already notified.
Transaction 11 was already notified.
Transaction 95 was already notified.
Trans

In [12]:
new_alerts = pd.DataFrame(
    new_alert_records
)

if not new_alerts.empty:

    if alert_history.empty:

        alert_history = new_alerts.copy()

    else:

        alert_history = pd.concat(
            [
                alert_history,
                new_alerts
            ],
            ignore_index=True
        )

    alert_history.to_csv(
        ALERT_HISTORY_FILE,
        index=False
    )

    print(
        "New notifications:",
        len(new_alerts)
    )

else:

    print(
        "No new notifications were required."
    )

print(
    "Total alert history:",
    len(alert_history)
)

display(
    alert_history.tail(10)
)

No new notifications were required.
Total alert history: 29


,alert_id,stream_id,alert_time,alert_type,risk_score,risk_level,notification_status
19,ALERT-20260815013252766990,96,2026-08-15T01:32:52.767031,Monitoring Alert,31.73,Medium,Desktop notification sent
20,ALERT-20260815013252767599,8,2026-08-15T01:32:52.767638,Monitoring Alert,31.49,Medium,Desktop notification sent
21,ALERT-20260815013252768233,59,2026-08-15T01:32:52.768265,Monitoring Alert,30.77,Medium,Desktop notification sent
22,ALERT-20260815013252768729,9,2026-08-15T01:32:52.768767,Monitoring Alert,30.73,Medium,Desktop notification sent
23,ALERT-20260815013252769332,98,2026-08-15T01:32:52.769370,Monitoring Alert,30.73,Medium,Desktop notification sent
24,ALERT-20260815013252769919,99,2026-08-15T01:32:52.769958,Monitoring Alert,30.62,Medium,Desktop notification sent
25,ALERT-20260815013252770503,11,2026-08-15T01:32:52.770541,Monitoring Alert,30.53,Medium,Desktop notification sent
26,ALERT-20260815013252771098,95,2026-08-15T01:32:52.771135,Monitoring Alert,30.35,Medium,Desktop notification sent
27,ALERT-20260815013252771646,25,2026-08-15T01:32:52.771683,Monitoring Alert,30.30,Medium,Desktop notification sent
28,ALERT-20260815013252772173,90,2026-08-15T01:32:52.772209,Monitoring Alert,30.23,Medium,Desktop notification sent


In [13]:
alert_history = pd.read_csv(
    "fraud_alert_history.csv"
)

print(
    "Before removing duplicates:",
    len(alert_history)
)

alert_history = (
    alert_history
    .drop_duplicates(
        subset=[
            "stream_id",
            "alert_type"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)

alert_history.to_csv(
    "fraud_alert_history.csv",
    index=False
)

print(
    "After removing duplicates:",
    len(alert_history)
)

Before removing duplicates: 29
After removing duplicates: 29


In [14]:
notification_summary = (
    alert_history
    .groupby(
        [
            "risk_level",
            "notification_status"
        ],
        observed=False
    )
    .size()
    .reset_index(
        name="total_alerts"
    )
)

notification_summary

,risk_level,notification_status,total_alerts
0,Medium,Desktop notification sent,29


In [15]:
alert_data.to_csv(
    "current_fraud_notifications.csv",
    index=False
)

notification_summary.to_csv(
    "fraud_notification_summary.csv",
    index=False
)

print("Files saved successfully:")
print("1. fraud_alert_history.csv")
print("2. current_fraud_notifications.csv")
print("3. fraud_notification_summary.csv")

Files saved successfully:
1. fraud_alert_history.csv
2. current_fraud_notifications.csv
3. fraud_notification_summary.csv
